# 실습 9: Ollama LLM을 활용한 HTTP 분류 (2교시)

특성 추출 없이 **HTTP 요청 텍스트를 그대로** Ollama gemma3:4b에 보여주고
정상/공격을 분류합니다.

**사전 조건**:
- Ollama 서버 실행 중 (`ollama serve`)
- `gemma3:4b` 모델 다운로드됨 (`ollama list`로 확인)
- 7주차 `processed_data.pkl` 존재 (LLM용 텍스트 샘플 포함)


In [3]:
# %% [Setup] 패키지 import 및 LLM용 샘플 로드
import pickle
import time
import json
import re
import pandas as pd
from urllib.parse import unquote
from sklearn.metrics import accuracy_score, f1_score, classification_report

import ollama  # pip install ollama

with open("processed_data.pkl", "rb") as f:
    data = pickle.load(f)

llm_sample = data["llm_sample"].head(100).reset_index(drop=True)
print(f"분류 대상: {len(llm_sample)}건")
print(f"라벨 분포: 정상 {(llm_sample.get('is_attack',0)==0).sum()}건 / "
      f"공격 {(llm_sample.get('is_attack',0)==1).sum()}건")


분류 대상: 100건
라벨 분포: 정상 43건 / 공격 57건


## 1. 분류 프롬프트 설계

LLM 응답을 안정적으로 파싱하기 위해:
- **Few-shot 예시** 2개로 출력 형식을 학습시킴
- **JSON 형태**로 응답하도록 강제
- 영어 프롬프트(gemma3:4b가 영어에 더 정확)

In [41]:
# %% [1] HTTP 텍스트 재구성 + 프롬프트 함수
def build_http_text(row) -> str:
    method = row.get("method", "GET")
    url    = unquote(str(row.get("url", "")), encoding="latin-1")
    body   = str(row.get("body_decoded", row.get("body", "")) or "")
    text   = f"{method} {url} HTTP/1.1"
    if body and body != "nan":
        text += f"\nBody: {body[:200]}"
    return text


PROMPT_TEMPLATE = PROMPT_TEMPLATE_FINAL = """You are a Web Security Guard. 
Classify the Request as "Normal" or "Anomalous".

[Checklist for "Anomalous"]
- Does it have special characters like: ' (quote), -- (dash), < > (brackets), ; (semicolon)?
- Does it have attack words like: SELECT, UNION, OR 1=1, ALERT, SCRIPT, ../ ?
- Is the URL path or body looking messy with symbols?

[Rule]
- If it is clean text/numbers (like id=10, name=james): Label as "Normal".
- If it matches ANY checklist above: Label as "Anomalous".

Request: {http_text}

Output format (Strict JSON):
{{"label": "Normal/Anomalous", "reason": "reason"}}"""


def classify_with_llm(http_text: str, model: str = "gemma3:4b") -> dict:
    """Ollama로 HTTP 요청 분류 -> {label, reason}"""
    prompt = PROMPT_TEMPLATE.format(http_text=http_text)
    response = ollama.chat(
        model=model,
        messages=[{"role":"user","content":prompt}],
        options={"temperature": 0},  # 결정성 높이기
    )
    text = response["message"]["content"]

    # JSON 추출 - LLM이 가끔 앞뒤 설명을 붙임
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        return {"label":"Unknown", "reason": text[:80]}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return {"label":"Unknown", "reason": text[:80]}


# 단건 테스트
test_text = "GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1"
print("입력:", test_text)
print("응답:", classify_with_llm(test_text))


입력: GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1
응답: {'label': 'Anomalous', 'reason': "The request contains special characters (' and '+OR+') and attack words ('OR') within the URL path and query parameters, indicating a potential SQL injection attempt."}


## 2. 100건 분류 + 시간 측정

CPU 환경 기준 건당 1~3초가 표준. 100건 ≈ 2~5분 소요.

In [42]:
# %% [2] 100건 분류
results = []
start = time.time()

for i, row in llm_sample.iterrows():
    http_text = build_http_text(row)
    result = classify_with_llm(http_text)
    true_label = "Anomalous" if row.get("is_attack", 0) == 1 else "Normal"
    results.append({
        "idx": i,
        "true": true_label,
        "pred": result.get("label", "Unknown"),
        "reason": result.get("reason", "")[:120],
        "http_short": http_text[:100],
    })
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"  {i+1}/{len(llm_sample)}건 완료 "
              f"({elapsed:.1f}초, 건당 {elapsed/(i+1):.2f}초)")

llm_time = time.time() - start
llm_df = pd.DataFrame(results)
print(f"\n총 소요: {llm_time:.1f}초")
print(f"1만 건 환산: 약 {llm_time/100*10000/60:.0f}분")


  10/100건 완료 (8.9초, 건당 0.89초)
  20/100건 완료 (17.8초, 건당 0.89초)
  30/100건 완료 (26.9초, 건당 0.90초)
  40/100건 완료 (36.4초, 건당 0.91초)
  50/100건 완료 (45.6초, 건당 0.91초)
  60/100건 완료 (54.1초, 건당 0.90초)
  70/100건 완료 (62.9초, 건당 0.90초)
  80/100건 완료 (71.5초, 건당 0.89초)
  90/100건 완료 (80.6초, 건당 0.90초)
  100/100건 완료 (89.1초, 건당 0.89초)

총 소요: 89.1초
1만 건 환산: 약 149분


In [43]:
# %% [3] 정확도/F1 계산
llm_df["pred_clean"] = llm_df["pred"].replace({"Unknown":"Normal"})
y_true = (llm_df["true"] == "Anomalous").astype(int)
y_pred = (llm_df["pred_clean"] == "Anomalous").astype(int)

llm_acc = accuracy_score(y_true, y_pred)
llm_f1  = f1_score(y_true, y_pred)

print(f"LLM 정확도: {llm_acc:.4f}")
print(f"LLM F1:    {llm_f1:.4f}")
print(f"분류 실패(Unknown): {(llm_df['pred']=='Unknown').sum()}건")
print()
print(classification_report(y_true, y_pred, target_names=["Normal","Anomalous"]))


LLM 정확도: 0.5800
LLM F1:    0.7308
분류 실패(Unknown): 0건

              precision    recall  f1-score   support

      Normal       1.00      0.02      0.05        43
   Anomalous       0.58      1.00      0.73        57

    accuracy                           0.58       100
   macro avg       0.79      0.51      0.39       100
weighted avg       0.76      0.58      0.44       100



## 3. 자연어 판단 근거 검토 ★

LLM의 가장 큰 강점: **왜 그렇게 판단했는지** 사람이 읽을 수 있는 문장으로 설명.
이는 SOC(보안관제) 분석가가 1차 분류를 검토할 때 매우 유용합니다.

In [9]:
# %% [4] 공격으로 판단한 사례 + LLM 근거
print("=== LLM이 공격으로 판단한 사례 (상위 5건) ===\n")
attack_pred = llm_df[llm_df["pred"] == "Anomalous"].head(5)
for _, r in attack_pred.iterrows():
    correct = "OK" if r["true"] == "Anomalous" else "오탐"
    print(f"[{correct}] 실제={r['true']:10s}  요청: {r['http_short']}")
    print(f"   - LLM 근거: {r['reason']}\n")


=== LLM이 공격으로 판단한 사례 (상위 5건) ===

[오탐] 실제=Normal      요청: GET /tienda1/miembros/editar.jsp?modo=registro&login=vilhelmi&password=elÁSTIcA&nombre=Theo&apellido
   - LLM 근거: The request contains a large number of parameters, including unusual ones like 'ntc' and 'B1', potentially indicating an

[오탐] 실제=Normal      요청: GET /tienda1/publico/registro.jsp?modo=registro&login=charmion&password=eu6eo&nombre=Amalio&apellido
   - LLM 근거: The request contains a large number of parameters, including unusual ones like 'ntc' and 'B1', combined with seemingly r

[OK] 실제=Anomalous   요청: GET /tienda1/miembros/editar.jsp/asf-logo-wide.gif.OLD HTTP/1.1
   - LLM 근거: The URL path includes a potentially malicious filename extension (.OLD) appended to a valid path, suggesting a possible 

[오탐] 실제=Normal      요청: GET /tienda1/publico/anadir.jsp?id=1&nombre=Jamón+Ibérico&precio=39&cantidad=73&B1=Añadir+al+carrito
   - LLM 근거: The request contains a parameter 'B1' which doesn't align with typical e-commerce flo

In [10]:
# %% [5] 결과 저장 (3교시에서도 활용)
with open("llm_classification_results.pkl", "wb") as f:
    pickle.dump({
        "llm_df": llm_df,
        "llm_acc": llm_acc,
        "llm_f1": llm_f1,
        "llm_time": llm_time,
        "n_samples": len(llm_sample),
    }, f)
print(">> llm_classification_results.pkl 저장 완료")


>> llm_classification_results.pkl 저장 완료


**다음**: `comparison_analysis.ipynb`로 1교시 ML 결과와 종합 비교합니다.